# OpenET vs NLDAS Noah ET Harmonization — Iowa 2019–2023

Computes the monthly delta between OpenET (actual ET including irrigation)
and NLDAS Noah ET (modeled ET without irrigation), masked to agricultural
pixels using the USDA Cropland Data Layer (CDL), to isolate the human
contribution to evapotranspiration in Iowa.

**Delta = OpenET − NLDAS_Noah_ET** (agricultural pixels only)
- Positive values → more ET than the land surface model predicts → likely irrigation
- Negative values → less ET than expected → drought, crop stress, or fallow land

**Data sources:**
- OpenET: `data/raw/OpenET/OpenET_Iowa_YYYYMM.tif` (mm/month, EPSG:4326, 0.125°)
- NLDAS Noah: `data/raw/NLDAS_Noah/NLDAS_NOAH0125_M.AYYYYMM.020.nc` — `EVPsfc` (mm/month, EPSG:4326, 0.125°)
- CDL: `data/raw/Cornbelt_annual_CDL/Cornbelt_annual_CDL_cropland_YYYY-01.tif` (EPSG:4326, ~0.0045°)

**Pipeline:**
1. Resample annual CDL to NLDAS 0.125° grid → cropland fraction per pixel
2. Build binary cropland mask (fraction > threshold)
3. For each month: load NLDAS + OpenET, align grids, apply mask, compute delta
4. Save masked delta GeoTIFFs and summary statistics

**Output:** `data/processed/ET_delta/ET_delta_YYYYMM_Iowa_cropland.tif`

In [ ]:
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 1. Paths and Configuration

In [ ]:
project_root = Path('../../..').resolve()

nldas_dir   = project_root / 'data' / 'raw' / 'NLDAS_Noah'
openet_dir  = project_root / 'data' / 'raw' / 'OpenET'
cdl_dir     = project_root / 'data' / 'raw' / 'Cornbelt_annual_CDL'
output_dir  = project_root / 'data' / 'processed' / 'ET_delta'
figures_dir = project_root / 'figures' / 'et_delta'
iowa_boundary = project_root / 'data' / 'aoi' / 'iowa.geojson'

output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

YEARS  = list(range(2019, 2024))
MONTHS = list(range(1, 13))

# CDL codes < 100 are crops; >= 100 are non-agricultural (water, developed, forest, etc.)
CROPLAND_MAX_CODE = 100

# Minimum cropland fraction within an NLDAS pixel to be included in analysis
CROPLAND_FRACTION_THRESHOLD = 0.5

# Verify input files
nldas_files  = sorted(nldas_dir.glob('NLDAS_NOAH0125_M.A*.nc'))
openet_files = sorted(openet_dir.glob('OpenET_Iowa_*.tif'))
cdl_files    = sorted(cdl_dir.glob('Cornbelt_annual_CDL_cropland_*.tif'))

print(f'NLDAS Noah files:  {len(nldas_files)} (expected 60)')
print(f'OpenET files:      {len(openet_files)} (expected 60)')
print(f'CDL files:         {len(cdl_files)}')
print(f'Output directory:  {output_dir}')

## 2. Verify Spatial Alignment

Confirm NLDAS, OpenET, and CDL are all in EPSG:4326 and check resolutions.

In [ ]:
iowa_gdf = gpd.read_file(iowa_boundary)

# NLDAS grid
ds_nldas = xr.open_dataset(nldas_files[0])
print('NLDAS Noah:')
print(f'  lon: {float(ds_nldas.lon[0]):.4f} to {float(ds_nldas.lon[-1]):.4f}, step={float(ds_nldas.lon[1]-ds_nldas.lon[0]):.4f}°')
print(f'  lat: {float(ds_nldas.lat[0]):.4f} to {float(ds_nldas.lat[-1]):.4f}, step={float(ds_nldas.lat[1]-ds_nldas.lat[0]):.4f}°')
ds_nldas.close()

# OpenET grid
with rasterio.open(openet_files[0]) as src:
    openet_transform = src.transform
    openet_crs = src.crs
    openet_shape = (src.height, src.width)
    print(f'\nOpenET:')
    print(f'  CRS: {openet_crs}')
    print(f'  Origin: ({openet_transform.c:.4f}, {openet_transform.f:.4f})')
    print(f'  Resolution: {openet_transform.a:.4f}°')
    print(f'  Shape: {openet_shape[0]} x {openet_shape[1]}')

# CDL
with rasterio.open(cdl_files[0]) as src:
    print(f'\nCDL:')
    print(f'  CRS: {src.crs}')
    print(f'  Resolution: {src.res[0]:.6f}° (~{src.res[0]*111:.1f} km)')
    print(f'  Shape: {src.height} x {src.width}')
    print(f'  → Will be resampled to 0.125° to match NLDAS/OpenET grid')

## 3. Define NLDAS Reference Grid and Build Annual Cropland Masks

NLDAS is the coarser reference grid (0.125°). OpenET will be resampled onto it.
CDL (~0.0045°) is also resampled to NLDAS resolution.

For each year:
1. Load CDL (crop codes, EPSG:4326, ~0.0045°)
2. Create binary cropland layer (code < 100 = crop)
3. Resample to NLDAS 0.125° grid using **average** resampling → cropland fraction per pixel
4. Threshold at `CROPLAND_FRACTION_THRESHOLD` to create final binary mask

In [ ]:
import rasterio.transform

def build_cropland_mask(cdl_path, target_transform, target_shape, target_crs='EPSG:4326',
                         threshold=0.5, cropland_max_code=100):
    """
    Resample CDL to NLDAS grid and return a boolean cropland mask.

    Returns
    -------
    mask : np.ndarray (bool), shape = target_shape
        True where the NLDAS pixel is >= threshold cropland fraction.
    crop_fraction : np.ndarray (float), shape = target_shape
        Cropland fraction per NLDAS pixel (0–1).
    """
    with rasterio.open(cdl_path) as src:
        cdl_data = src.read(1).astype(np.float32)
        cdl_transform = src.transform
        cdl_crs = src.crs

    # Binary: 1 = cropland (code < 100), 0 = non-crop
    binary = np.where((cdl_data > 0) & (cdl_data < cropland_max_code), 1.0, 0.0)
    binary[cdl_data == 0] = np.nan  # true nodata → NaN so it doesn't dilute the fraction

    # Resample to NLDAS grid using average → gives cropland fraction per pixel
    crop_fraction = np.empty(target_shape, dtype=np.float32)
    reproject(
        source=binary,
        destination=crop_fraction,
        src_transform=cdl_transform,
        src_crs=cdl_crs,
        src_nodata=np.nan,
        dst_transform=target_transform,
        dst_crs=target_crs,
        resampling=Resampling.average
    )

    mask = crop_fraction >= threshold
    return mask, crop_fraction


# ── Get NLDAS reference grid (transform + shape) ──────────────────────────
# NLDAS pixel centers: lon from -124.9375 to -67.0625, lat from 25.0625 to 52.9375
# Pixel origin (upper-left corner) = center - half_res
ds_ref = xr.open_dataset(nldas_files[0])
nldas_lons = ds_ref.lon.values
nldas_lats = ds_ref.lat.values
ds_ref.close()

nldas_res = 0.125
nldas_transform = rasterio.transform.from_origin(
    west  = nldas_lons[0]  - nldas_res / 2,   # left edge of first pixel
    north = nldas_lats[-1] + nldas_res / 2,    # top edge of last (northernmost) lat
    xsize = nldas_res,
    ysize = nldas_res
)
nldas_shape = (len(nldas_lats), len(nldas_lons))

print(f'NLDAS reference grid:')
print(f'  Shape: {nldas_shape[0]} rows x {nldas_shape[1]} cols')
print(f'  Transform: {nldas_transform}')

# ── Build cropland masks for each year ───────────────────────────────────
cropland_masks    = {}
cropland_fraction = {}

for year in YEARS:
    cdl_path = cdl_dir / f'Cornbelt_annual_CDL_cropland_{year}-01.tif'
    if not cdl_path.exists():
        print(f'  WARNING: CDL not found for {year}')
        continue
    mask, frac = build_cropland_mask(
        cdl_path, nldas_transform, nldas_shape,
        threshold=CROPLAND_FRACTION_THRESHOLD
    )
    cropland_masks[year]    = mask
    cropland_fraction[year] = frac
    pct = mask.sum() / mask.size * 100
    print(f'  {year}: {mask.sum()} cropland pixels ({pct:.1f}% of NLDAS grid)  '
          f'| mean cropland fraction = {np.nanmean(frac):.2f}')

print('\nCropland masks built.')

## 4. Visualize Cropland Masks

In [ ]:
fig, axes = plt.subplots(1, len(YEARS), figsize=(20, 4))

for ax, year in zip(axes, YEARS):
    if year not in cropland_fraction:
        ax.set_visible(False)
        continue
    im = ax.imshow(
        cropland_fraction[year],
        extent=[openet_transform.c,
                openet_transform.c + openet_transform.a * target_shape[1],
                openet_transform.f + openet_transform.e * target_shape[0],
                openet_transform.f],
        origin='upper', cmap='YlGn', vmin=0, vmax=1
    )
    iowa_gdf.boundary.plot(ax=ax, color='black', linewidth=0.8)
    n = cropland_masks[year].sum()
    ax.set_title(f'{year}\n{n} crop pixels', fontsize=10, fontweight='bold')
    ax.set_xlabel('Lon')

axes[0].set_ylabel('Lat')
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.2)
cbar.set_label(f'Cropland fraction per 0.125° pixel  |  mask threshold = {CROPLAND_FRACTION_THRESHOLD}')
plt.suptitle('Annual CDL Cropland Fraction — Iowa (resampled to NLDAS 0.125° grid)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figures_dir / 'cropland_masks.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Process All 60 Months — Compute Masked Delta

In [ ]:
records     = []
delta_grids = {}  # (year, month) → masked delta DataArray
skipped     = []

for year in YEARS:
    mask = cropland_masks.get(year)
    if mask is None:
        print(f'  SKIP {year}: no cropland mask')
        skipped.extend([f'{year}{m:02d}' for m in MONTHS])
        continue

    for month in MONTHS:
        yyyymm = f'{year}{month:02d}'

        nldas_path  = nldas_dir  / f'NLDAS_NOAH0125_M.A{yyyymm}.020.nc'
        openet_path = openet_dir / f'OpenET_Iowa_{yyyymm}.tif'
        out_path    = output_dir / f'ET_delta_{yyyymm}_Iowa_cropland.tif'

        if not nldas_path.exists() or not openet_path.exists():
            skipped.append(yyyymm)
            print(f'  SKIP {yyyymm}: missing file(s)')
            continue

        # ── Load NLDAS Noah EVPsfc (reference grid) ──────────────────────
        ds = xr.open_dataset(nldas_path)
        nldas_et = ds['EVPsfc'].squeeze()          # mm/month, EPSG:4326, 0.125°
        nldas_et = nldas_et.rename({'lon': 'x', 'lat': 'y'})
        nldas_et = nldas_et.rio.write_crs('EPSG:4326')
        nldas_iowa = nldas_et.rio.clip(iowa_gdf.geometry, iowa_gdf.crs, drop=True)
        ds.close()

        # ── Load OpenET and resample onto NLDAS grid ──────────────────────
        # OpenET is higher resolution — reproject_match resamples it to
        # exactly match NLDAS pixel grid (same CRS, transform, shape)
        openet_raw = rxr.open_rasterio(openet_path, masked=True).squeeze()
        openet_resampled = openet_raw.rio.reproject_match(
            nldas_iowa,
            resampling=Resampling.average
        )

        # ── Compute raw delta on NLDAS grid ──────────────────────────────
        delta = openet_resampled.values - nldas_iowa.values   # mm/month

        # ── Clip cropland mask to Iowa NLDAS extent ───────────────────────
        # nldas_iowa is clipped to Iowa; mask covers full CONUS NLDAS grid
        # Extract the Iowa rows/cols from the full mask using NLDAS coordinates
        iowa_lons = nldas_iowa.x.values
        iowa_lats = nldas_iowa.y.values
        lon_idx = np.searchsorted(nldas_lons, iowa_lons)
        lat_idx = np.searchsorted(nldas_lats, iowa_lats)
        iowa_mask = mask[np.ix_(lat_idx, lon_idx)]

        if iowa_mask.shape != delta.shape:
            raise ValueError(f'{yyyymm}: mask shape {iowa_mask.shape} != delta shape {delta.shape}')

        # ── Apply cropland mask ───────────────────────────────────────────
        delta_masked = np.where(iowa_mask, delta, np.nan)

        # ── Save GeoTIFF on NLDAS grid ────────────────────────────────────
        delta_da = xr.DataArray(
            delta_masked,
            coords={'y': nldas_iowa.y, 'x': nldas_iowa.x},
            dims=['y', 'x']
        )
        delta_da = delta_da.rio.write_crs('EPSG:4326')
        delta_da.rio.to_raster(out_path)

        # ── Summary stats (cropland pixels only) ─────────────────────────
        valid_openet = openet_resampled.values[iowa_mask & ~np.isnan(openet_resampled.values)]
        valid_nldas  = nldas_iowa.values[iowa_mask & ~np.isnan(nldas_iowa.values)]
        valid_delta  = delta_masked[~np.isnan(delta_masked)]

        records.append({
            'year': year, 'month': month, 'yyyymm': yyyymm,
            'n_crop_pixels': int(iowa_mask.sum()),
            'openet_mean':   float(valid_openet.mean()) if len(valid_openet) else np.nan,
            'nldas_mean':    float(valid_nldas.mean())  if len(valid_nldas)  else np.nan,
            'delta_mean':    float(valid_delta.mean())  if len(valid_delta)  else np.nan,
            'delta_std':     float(valid_delta.std())   if len(valid_delta)  else np.nan,
            'delta_min':     float(valid_delta.min())   if len(valid_delta)  else np.nan,
            'delta_max':     float(valid_delta.max())   if len(valid_delta)  else np.nan,
        })
        delta_grids[(year, month)] = delta_da

        print(f'  {yyyymm}: OpenET={records[-1]["openet_mean"]:6.1f}  '
              f'NLDAS={records[-1]["nldas_mean"]:6.1f}  '
              f'Δ={records[-1]["delta_mean"]:+6.1f} mm/month  '
              f'[{records[-1]["n_crop_pixels"]} crop px]')

df = pd.DataFrame(records)
print(f'\nProcessed {len(df)} months, skipped {len(skipped)}')

## 6. Save Summary CSV

In [ ]:
csv_path = output_dir / 'ET_delta_monthly_summary_Iowa_cropland.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
print(df.to_string(index=False))

## 7. Visualize — Time Series of Mean Delta (Cropland Only)

In [ ]:
dates = pd.to_datetime(df['yyyymm'], format='%Y%m')

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Top: OpenET vs NLDAS Noah
axes[0].plot(dates, df['openet_mean'], 'o-', color='steelblue', label='OpenET (actual)', linewidth=1.5)
axes[0].plot(dates, df['nldas_mean'],  'o-', color='darkorange', label='NLDAS Noah (modeled)', linewidth=1.5)
axes[0].set_ylabel('Mean ET — cropland pixels (mm/month)', fontsize=11)
axes[0].set_title('OpenET vs NLDAS Noah ET — Iowa Cropland 2019–2023', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: Delta
bar_colors = ['tomato' if v > 0 else 'steelblue' for v in df['delta_mean']]
axes[1].bar(dates, df['delta_mean'], width=20, color=bar_colors, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].fill_between(dates,
    df['delta_mean'] - df['delta_std'],
    df['delta_mean'] + df['delta_std'],
    alpha=0.2, color='gray', label='±1 std dev')
axes[1].set_ylabel('ΔET = OpenET − NLDAS (mm/month)', fontsize=11)
axes[1].set_title('Human ET Contribution (ΔET) — Cropland Pixels Only\nRed = Excess ET (irrigation signal)',
                  fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for year in YEARS:
    axes[1].axvline(pd.Timestamp(f'{year}-01-01'), color='gray', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_timeseries_cropland.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Visualize — Seasonal Cycle of Delta

In [ ]:
colors_yr = {2019: '#1f77b4', 2020: '#ff7f0e', 2021: '#2ca02c', 2022: '#d62728', 2023: '#9467bd'}

fig, ax = plt.subplots(figsize=(12, 6))

for year in YEARS:
    yr = df[df['year'] == year].sort_values('month')
    if not yr.empty:
        ax.plot(yr['month'], yr['delta_mean'], 'o-',
                color=colors_yr[year], linewidth=2, markersize=6, label=str(year))

monthly_mean = df.groupby('month')['delta_mean'].mean()
ax.plot(monthly_mean.index, monthly_mean.values, 'k--', linewidth=2.5, label='5-yr mean', zorder=5)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('ΔET (mm/month)', fontsize=12)
ax.set_title('Seasonal Cycle of Human ET Contribution — Iowa Cropland\nΔET = OpenET − NLDAS Noah',
             fontsize=13, fontweight='bold')
ax.legend(title='Year')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_seasonal_cycle_cropland.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visualize — Spatial Maps of Masked Delta

In [ ]:
sample_year = 2021
plot_months = [4, 6, 7, 8, 10]
month_names = {4:'April', 6:'June', 7:'July', 8:'August', 10:'October'}

fig, axes = plt.subplots(1, len(plot_months), figsize=(20, 5))

all_vals = np.concatenate([
    delta_grids[(sample_year, m)].values.flatten()
    for m in plot_months if (sample_year, m) in delta_grids
])
vmax = np.nanpercentile(np.abs(all_vals), 95)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for ax, month in zip(axes, plot_months):
    if (sample_year, month) not in delta_grids:
        ax.set_visible(False)
        continue
    data = delta_grids[(sample_year, month)]
    im = ax.imshow(
        data.values,
        extent=[float(data.x.min()), float(data.x.max()),
                float(data.y.min()), float(data.y.max())],
        origin='upper', cmap='RdBu_r', norm=norm
    )
    iowa_gdf.boundary.plot(ax=ax, color='black', linewidth=0.8)
    mean_val = float(np.nanmean(data.values))
    ax.set_title(f'{month_names[month]} {sample_year}\nΔ={mean_val:+.1f} mm', fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.15)
cbar.set_label('ΔET = OpenET − NLDAS Noah (mm/month)  |  Red = Excess ET  |  Grey = Non-cropland (masked)', fontsize=9)
plt.suptitle(f'Human ET Contribution — Iowa Cropland {sample_year}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figures_dir / f'et_delta_maps_{sample_year}_cropland.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Annual Mean Delta Heatmap

In [ ]:
pivot = df.pivot(index='year', columns='month', values='delta_mean')

vmax = np.nanpercentile(np.abs(pivot.values[~np.isnan(pivot.values)]), 95)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot.values, cmap='RdBu_r', norm=norm, aspect='auto')

ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(12):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.0f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > vmax * 0.5 else 'black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('ΔET (mm/month)')
ax.set_title('ΔET Heatmap — Iowa Cropland 2019–2023  |  Red = Excess ET (irrigation signal)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_heatmap_cropland.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary

In [ ]:
month_names_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('=' * 65)
print('ET Harmonization Summary — Iowa Cropland 2019–2023')
print('=' * 65)
print(f'Cropland fraction threshold: {CROPLAND_FRACTION_THRESHOLD}')
print(f'Months processed: {len(df)}')
print(f'Months skipped:   {len(skipped)}')
print()
print(f'Mean cropland pixels per month: {df["n_crop_pixels"].mean():.0f}')
print(f'Overall mean OpenET (cropland): {df["openet_mean"].mean():.1f} mm/month')
print(f'Overall mean NLDAS  (cropland): {df["nldas_mean"].mean():.1f} mm/month')
print(f'Overall mean ΔET    (cropland): {df["delta_mean"].mean():+.1f} mm/month')
print()
print('Annual mean ΔET (cropland pixels):')
for year, grp in df.groupby('year'):
    print(f'  {year}: {grp["delta_mean"].mean():+.1f} mm/month  '
          f'(annual sum: {grp["delta_mean"].sum():+.0f} mm/yr)')
print()
print('Peak months (highest mean ΔET):')
top = df.groupby('month')['delta_mean'].mean().sort_values(ascending=False).head(4)
for m, v in top.items():
    print(f'  {month_names_map[m]}: {v:+.1f} mm/month')
print()
print(f'Output GeoTIFFs: {output_dir}')
print(f'Summary CSV:     {csv_path}')
print(f'Figures:         {figures_dir}')